# 그림 스토리 단어 미션 — 이야기 AI 파인튜닝 4차 · E2B + 모범 이야기 1,200편 (실험)

E4B는 메모리 8GB 폰(갤럭시 노트20)에서 그래픽 칩 메모리 한도에 걸려 올라가지 않았어요. 그래서 4차는 **지금 게임이 쓰는 Gemma 4 E2B**를
3차(모범 300편)와 **같은 형식의 모범 이야기 1,200편**(`tools/finetune/v3/train.jsonl`)으로 다시 가르쳐요.
이야기 수를 4배로 늘리면 E2B가 얼마나 나아지는지 보는 실험이에요 (3차 E2B: 형식 9/30, 쓸 만한 이야기 3~4편).

순서: 학습 → 학습 파일(LoRA) 올리기 → **재 보기 30편** 올리기 → 앱이 크롬에서 쓰는 꼴(MLC q4f16_1)로 바꿔 올리기. 모두 내 허깅페이스 계정에 공개로 올려요.

**중간 저장**: 50걸음마다 허깅페이스 `checkpoint` 폴더에 저장해요. 끊기면 **노트북을 다시 열고 '모두 실행'**만 하면 저장한 곳부터 이어서 해요
(설치·모델 받기는 다시 해요). 재 보기 30편을 이미 올렸으면 다시 쓰지 않고 크롬용 바꾸기로 넘어가요.

**할 일**
1. 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 고르고 저장.
2. 왼쪽 **열쇠 모양(보안 비밀)**에 이름 `HF_TOKEN`, 값은 허깅페이스 **쓰기(write)** 토큰, **노트북 액세스** 켜기 (3차 때 넣었으면 그대로 있어요).
   휴대폰이라 열쇠 모양을 못 찾으면 건너뛰어요: 3번 셀에 입력 칸이 나오면 거기에 붙여 넣고 Enter (글자는 가려지고 저장되지 않아요).
3. **런타임 → 모두 실행**. 3번 셀에서 **구글 드라이브 연결**을 물으면 허용해 주세요: 재 보기 결과 파일 하나(`wordstory4-e2b-eval.json`)를 '내 드라이브'에 저장해요. 거절해도 학습은 그대로 돼요.
4. **창을 앞에 띄워 둔 채로** 두세요 (휴대폰은 화면을 켜 두고 충전기에). 학습 1시간~1시간 반, 재 보기 20분쯤, 크롬용 바꾸기 20~30분: 모두 2시간~2시간 반쯤.
5. 끝에 나오는 주소를 Claude에게 알려 주세요 (7번 '재 보기'까지만 끝나도 Claude가 잴 수 있어요).

## 1. 그래픽 칩 확인과 설정

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU가 없어요: 런타임 → 런타임 유형 변경 → T4 GPU로 바꾸고 다시 '모두 실행' 해 주세요"
print("GPU:", torch.cuda.get_device_name(0))

TRAIN = True          # False: 학습·재 보기는 건너뛰고, 이미 올린 학습 파일(lora-adapter)로 크롬용 모델만 다시 만들어요
MAKE_MLC = True       # 재 보기 뒤에 앱이 크롬에서 쓰는 꼴로 바꿔 올려요
MERGE_SCALE = 1.0     # 학습한 만큼을 얼마나 세게 넣을지 (Claude가 재 보고 알려 줘요. 보통 1.0)
BRANCH = "claude/gracious-cannon-7stm9g"
REPO_URL = "https://github.com/areweloststars/-.git"
DATA = "repo/tools/finetune/v3"
BASE = "google/gemma-4-E2B-it"
BASE_REV = "3e22461f65e89153144f8adb70e3b8c2cc9845a7"  # 앱의 크롬용 모델을 만든 판
MLC_BASE = "mlc-ai/gemma-4-E2B-it-q4f16_1-MLC"        # 앱이 지금 쓰는 크롬용 모델
OUT_NAME = "gemma-4-E2B-it-wordstory4-q4f16_1-MLC"    # 내 허깅페이스에 만들 저장소 이름
MIN_ROWS = 1200   # 모범 이야기가 이만큼 안 되면 학습하지 않아요
EPOCHS = 2        # 1,200편 × 2번 ÷ 4 = 600걸음
SAVE_STEPS = 50   # 이만큼 걸을 때마다 허깅페이스 checkpoint 폴더에 중간 저장
LORA_R = 16
LR = 2e-4
TRIES = 3         # 재 보기: 시험 묶음마다 몇 편씩 쓸지 (3차 재기와 같아요)
EVAL_FILE = "wordstory4-e2b-eval.json"   # 구글 드라이브 '내 드라이브'에 저장할 재 보기 결과

## 2. 학습 도구 설치 (Unsloth 공식 Gemma 4 노트북과 같은 설치)

In [ ]:
%%capture
import os
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
import importlib.metadata as _md
try: _tv = tuple(int(x) for x in _md.version("torch").split("+")[0].split(".")[:2])
except Exception: _tv = (2, 11)
_torchcodec = {(2, 11): "torchcodec>=0.11.0", (2, 10): "torchcodec==0.10.*", (2, 9): "torchcodec==0.9.*", (2, 8): "torchcodec==0.7.*", (2, 7): "torchcodec==0.5.*"}.get(_tv, "torchcodec>=0.12.0" if _tv >= (2, 12) else "torchcodec")
!pip install "{_torchcodec}"
!pip install --no-deps --upgrade timm
import torch; torch._dynamo.config.recompile_limit = 64;

## 3. 허깅페이스 토큰·중간 저장 확인, 모범 이야기와 크롬용 모델 받기
학습 전에 확인할 것을 먼저 확인해요 (여기서 멈추면 오래 기다리지 않아도 돼요).
허깅페이스에 중간 저장이 있으면 그때 쓴 이야기 파일(같은 커밋)을 받아 이어서 해요.

In [ ]:
import json, os, shutil, subprocess, sys
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    # 보안 비밀이 없으면 (휴대폰 등) 여기서 받아요. 글자는 가려지고 노트북에 저장되지 않아요
    from getpass import getpass
    print("보안 비밀 HF_TOKEN이 없어요 (" + type(e).__name__ + "). 아래 칸에 허깅페이스 쓰기 토큰을 붙여 넣고 Enter를 눌러 주세요.")
    HF_TOKEN = getpass("토큰: ").strip()
api = HfApi(token=HF_TOKEN)
me = api.whoami()
role = (me.get("auth") or {}).get("accessToken", {}).get("role")
assert role != "read", "읽기 토큰이에요. 허깅페이스에서 쓰기(write) 토큰을 새로 만들어 넣어 주세요"
REPO_ID = f"{me['name']}/{OUT_NAME}"
api.create_repo(REPO_ID, repo_type = "model", private = False, exist_ok = True)
print("올릴 곳:", "https://huggingface.co/" + REPO_ID)
FILES = api.list_repo_files(REPO_ID)

# 중간 저장이 있으면 그 걸음부터 이어서 해요 (이야기 파일도 그때 쓴 커밋 그대로)
RESUME, DATA_COMMIT = None, None
if "checkpoint/data.json" in FILES:
    info = json.load(open(hf_hub_download(REPO_ID, "checkpoint/data.json", token = HF_TOKEN)))
    assert info["epochs"] == EPOCHS, f"중간 저장은 EPOCHS = {info['epochs']}로 했어요. 1번 셀의 EPOCHS를 그대로 두고 다시 실행해 주세요"
    DATA_COMMIT = info["commit"]
    if TRAIN:
        RESUME = f"outputs/checkpoint-{info['step']}"
        for d in ("outputs", "resume"): shutil.rmtree(d, ignore_errors = True)
        snapshot_download(REPO_ID, allow_patterns = ["checkpoint/*"], local_dir = "resume", token = HF_TOKEN)
        os.makedirs("outputs")
        shutil.move("resume/checkpoint", RESUME)
        print(f"중간 저장을 찾았어요: {info['step']}/{info['max_steps']}걸음부터 이어서 해요")

def run(*cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

run("rm", "-rf", "repo")
run("git", "init", "-q", "repo")
run("git", "-C", "repo", "fetch", "-q", "--depth", "1", REPO_URL, DATA_COMMIT or BRANCH)
run("git", "-C", "repo", "checkout", "-q", "FETCH_HEAD")
DATA_COMMIT = subprocess.run(["git", "-C", "repo", "rev-parse", "HEAD"], capture_output = True, text = True, check = True).stdout.strip()
rows = [json.loads(l) for l in open(f"{DATA}/train.jsonl", encoding="utf-8") if l.strip()]
tests = json.load(open(f"{DATA}/test-prompts.json", encoding="utf-8"))
print("이야기 커밋", DATA_COMMIT[:7], "· 모범 이야기", len(rows), "편 · 재 볼 요청", len(tests), "개")
# 이어서 할 때는 처음 돌릴 때 받은 이야기(같은 커밋) 그대로 해요
assert not TRAIN or RESUME or len(rows) >= MIN_ROWS, f"모범 이야기가 아직 {len(rows)}편이에요 ({MIN_ROWS}편이 되면 돌려요). Claude에게 알려 주세요"

# 재 보기 30편을 같은 이야기 커밋으로 이미 다 올렸으면 다시 쓰지 않아요 (끊긴 뒤 다시 '모두 실행'했을 때)
EVAL_DONE = False
if "eval/outputs.json" in FILES:
    ev = json.load(open(hf_hub_download(REPO_ID, "eval/outputs.json", token = HF_TOKEN)))
    EVAL_DONE = ev.get("commit") == DATA_COMMIT and len(ev.get("runs", [])) >= TRIES * len(tests)
    if EVAL_DONE: print("재 보기 30편은 이미 올려 두었어요: 다시 쓰지 않아요")

# 재 보기 결과를 구글 드라이브에도 저장해요. 거절하면 허깅페이스에만 올려요
DRIVE = None
if TRAIN and not EVAL_DONE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", timeout_ms = 300000)
        DRIVE = "/content/drive/MyDrive"
        print("구글 드라이브 연결됨: 재 보기 결과를 '내 드라이브'에도 저장해요")
    except Exception as e:
        print("구글 드라이브 없이 해요 (" + type(e).__name__ + "). 재 보기 결과는 허깅페이스에만 올려요")

if MAKE_MLC:
    # 앱이 쓰는 크롬용 모델(약 3GB)을 받고, 변환 도구가 원본과 바이트까지 같게 만드는지 0층으로 확인해요
    snapshot_download(MLC_BASE, local_dir="mlc_base")
    base_url = f"https://huggingface.co/{BASE}/resolve/{BASE_REV}/model.safetensors"
    run(sys.executable, "repo/tools/finetune/mlc_patch.py", "--hf", base_url, "--mlc", "mlc_base", "--verify", "--only", "layers.0.")

## 4. 모델 받기 (약 10GB, 몇 분)
`TRAIN = False`이면 원본 대신 이미 올린 학습 파일을 함께 받아요.

In [ ]:
from unsloth import FastModel
if TRAIN:
    src = BASE
else:
    src = snapshot_download(REPO_ID, allow_patterns=["lora-adapter/*"], token=HF_TOKEN) + "/lora-adapter"
    print("학습 파일:", src)
model, tokenizer = FastModel.from_pretrained(
    model_name = src,
    dtype = None,
    max_seq_length = 1024,  # 모범 답안은 가장 긴 것이 대화 틀까지 620토큰
    load_in_4bit = False,
    full_finetuning = False,
    token = HF_TOKEN,
)
tok = getattr(tokenizer, "tokenizer", tokenizer)  # 글자 토크나이저
BOS = tok.bos_token_id
TURN_END = tok.convert_tokens_to_ids("<turn|>")
print("bos", BOS, "· <turn|>", TURN_END)

def chat(user, answer=None):
    # WebLLM gemma4_instruction과 Gemma 4 공식 대화 틀이 만드는 글과 같아요
    s = "<bos><|turn>user\n" + user + "<turn|>\n<|turn>model\n"
    return s if answer is None else s + answer + "<turn|>\n"

## 5. 학습 준비
앱의 크롬 모델(WebLLM `gemma4_instruction`)과 똑같은 대화 꼴로 쓰고, 답(이야기) 부분만 배우게 해요.
50걸음마다 중간 저장(`outputs/checkpoint-걸음`)을 허깅페이스 `checkpoint` 폴더로 올려요 (앞의 것은 지워요).

In [ ]:
if TRAIN:
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback
    from unsloth.chat_templates import train_on_responses_only

    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers = False,
        finetune_language_layers = True,
        finetune_attention_modules = True,
        finetune_mlp_modules = True,
        r = LORA_R,
        lora_alpha = LORA_R,
        lora_dropout = 0,
        bias = "none",
        random_state = 3407,
    )
    texts = [chat(r["messages"][0]["content"], r["messages"][1]["content"]) for r in rows]

    class SaveToHub(TrainerCallback):
        # 중간 저장을 허깅페이스에 올려요. 한 번 못 올려도 학습은 계속해요 (다음 저장 때 다시 올려요)
        def on_save(self, args, state, control, **kwargs):
            ck = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
            if not os.path.isdir(ck): return
            json.dump({"commit": DATA_COMMIT, "branch": BRANCH, "rows": len(rows), "epochs": EPOCHS,
                       "step": state.global_step, "max_steps": state.max_steps}, open(os.path.join(ck, "data.json"), "w"))
            try:
                api.upload_folder(folder_path = ck, path_in_repo = "checkpoint", repo_id = REPO_ID, delete_patterns = "*",
                                  commit_message = f"checkpoint {state.global_step}/{state.max_steps}")
                print(f"중간 저장 올림: {state.global_step}/{state.max_steps}걸음", flush = True)
            except Exception as e:
                print("중간 저장을 못 올렸어요:", type(e).__name__, e, flush = True)

    def make_trainer(texts):
        t = SFTTrainer(
            model = model,
            tokenizer = tokenizer,
            train_dataset = Dataset.from_dict({"text": texts}),
            eval_dataset = None,
            callbacks = [SaveToHub()],
            args = SFTConfig(
                dataset_text_field = "text",
                per_device_train_batch_size = 1,
                gradient_accumulation_steps = 4,
                warmup_steps = 5,
                num_train_epochs = EPOCHS,
                learning_rate = LR,
                logging_steps = 10,
                save_strategy = "steps",
                save_steps = SAVE_STEPS,
                save_total_limit = 1,
                optim = "adamw_8bit",
                weight_decay = 0.001,
                lr_scheduler_type = "linear",
                seed = 3407,
                output_dir = "outputs",
                report_to = "none",
            ),
        )
        return train_on_responses_only(t, instruction_part = "<|turn>user\n", response_part = "<|turn>model\n")

    trainer = make_trainer(texts)
    ids = list(trainer.train_dataset[0]["input_ids"])
    if ids[:2] == [BOS, BOS]:  # 토크나이저가 <bos>를 또 붙였으면 글에서 빼요
        trainer = make_trainer([t[len("<bos>"):] for t in texts])
        ids = list(trainer.train_dataset[0]["input_ids"])
    assert ids[0] == BOS and ids.count(BOS) == 1, "<bos>가 맨 앞에 하나여야 해요"
    assert TURN_END in ids, "<turn|>이 한 토큰이 아니에요"
    labels = trainer.train_dataset[0]["labels"]
    learned = tok.decode([i for i, l in zip(ids, labels) if l != -100])
    print("배울 부분 (첫 이야기):\n" + learned)
    assert learned.strip() and "단어 차례는" not in learned and "그림:" in learned, "답 부분만 배우게 하지 못했어요"
else:
    print("TRAIN = False: 학습은 건너뛰어요")

## 6. 학습 (T4에서 1시간~1시간 반쯤) → 학습 파일(LoRA)을 올려요
끊기면 노트북을 다시 열고 '모두 실행'하면 중간 저장부터 이어서 해요. 다 끝난 뒤 다시 돌려도 마지막 중간 저장에서 바로 끝나요.

In [ ]:
README = f"""---
license: apache-2.0
base_model: {BASE}
tags:
- mlc-llm
- web-llm
- gemma4
language:
- ko
---
# {OUT_NAME}

**Modified version** of [{BASE}](https://huggingface.co/{BASE}) (revision `{BASE_REV}`), licensed under Apache 2.0.

- Changes: LoRA fine-tuned (r={LORA_R}, {EPOCHS} epochs, merged at scale {MERGE_SCALE}, {len(rows)} short Korean children's stories written for this project
  with common dictionary words; stories: commit {DATA_COMMIT[:7]}) so that, given 6-8 English words with Korean meanings, it writes a title,
  a one-line plot, a short story with one `{{english|korean}}` word marker per word line (choosing the word order itself) and an English picture
  description for each word, for the word game in [areweloststars/-](https://github.com/areweloststars/-).
- Round 4: the same format as round 3 (300 example stories), with four times as many example stories.
- The merged weights are re-quantized into the file layout of [{MLC_BASE}](https://huggingface.co/{MLC_BASE}) (q4f16_1)
  with `tools/finetune/mlc_patch.py`. Only the text-layer linear weights (q/k/v/o, gate_up, down) differ from that repository;
  use it with the same WebLLM model library (`gemma-4-E2B-it-q4f16_1`).
- `lora-adapter/`: the LoRA adapter before merging. `checkpoint/`: the last training checkpoint (for resuming).
  `eval/outputs.json`: 30 sample outputs for the project's own checks.
"""
if TRAIN:
    stats = trainer.train(resume_from_checkpoint = RESUME)
    print(f"{stats.metrics['train_runtime'] / 60:.1f}분 · loss {stats.training_loss:.3f}")
    model.save_pretrained("lora")
    tokenizer.save_pretrained("lora")
    api.upload_folder(folder_path = "lora", path_in_repo = "lora-adapter", repo_id = REPO_ID, commit_message = "LoRA adapter")
    import urllib.request
    api.upload_file(path_or_fileobj = urllib.request.urlopen("https://www.apache.org/licenses/LICENSE-2.0.txt").read(), path_in_repo = "LICENSE", repo_id = REPO_ID)
    api.upload_file(path_or_fileobj = README.encode(), path_in_repo = "README.md", repo_id = REPO_ID)
    print("\n학습 파일을 올렸어요: https://huggingface.co/" + REPO_ID + "\n이어서 재 보기 30편을 써요 (20분쯤). 끝까지 기다려 주세요")
else:
    print("TRAIN = False: 학습은 건너뛰어요")

## 7. 재 보기 30편 (시험 묶음 10개 × 3편, 3차 재기와 같은 설정)
여기서는 꼴만 봐요(제목·줄거리·줄 수·단어 표시·그림 줄). 쓴 글을 허깅페이스 `eval/outputs.json`(구글 드라이브를 연결했으면 '내 드라이브'의 `wordstory4-e2b-eval.json`에도)에 저장해서,
**진짜 재기는 Claude가 앱의 점검으로** 해요. 묶음 하나가 끝날 때마다 저장해요.

In [ ]:
import re
TAG = re.compile(r"\{([^{|}]+)\|([^{|}]+)\}")

def write(prompt, n):
    x = tok(chat(prompt), add_special_tokens = False, return_tensors = "pt").to("cuda")
    out = model.generate(**x, max_new_tokens = 200 + 60 * (n + 2), do_sample = True, temperature = 0.7, top_p = 0.9,
                         eos_token_id = [tok.eos_token_id, TURN_END])
    return tok.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens = True).strip()

def parts(text):
    out, part = {"제목": "", "줄거리": "", "이야기": [], "그림": []}, ""
    for l in [re.sub(r"\*\*", "", l).strip() for l in text.split("\n")]:
        if not l: continue
        m = re.match(r"^(제목|줄거리|이야기|그림)\s*[:：]\s*(.*)$", l)
        if m:
            part = m.group(1)
            if part in ("제목", "줄거리"): out[part] = m.group(2)
            elif m.group(2): out[part].append(m.group(2))
            continue
        if part in ("이야기", "그림"): out[part].append(l)
    return out

def form(words, text):
    p, n = parts(text), len(words)
    if not p["제목"]: return "제목 없음"
    if not p["줄거리"]: return "줄거리 없음"
    lines = p["이야기"]
    if not (n + 2 <= len(lines) <= n + 4): return f"이야기 {len(lines)}줄 ({n + 2}~{n + 4}줄이어야 해요)"
    if TAG.search(lines[0]) or TAG.search(lines[-1]): return "첫 줄·끝 줄에는 단어 표시가 없어야 해요"
    got = [t[0].strip().lower() for l in lines for t in TAG.findall(l)]
    for w in words:
        if got.count(w[0].lower()) != 1: return f"{w[0]} 표시가 {got.count(w[0].lower())}번"
    pics = {re.split(r"\s*[:：]", l, 1)[0].strip().lower() for l in p["그림"]}
    miss = [w[0] for w in words if w[0].lower() not in pics]
    return f"그림 없음: {', '.join(miss)}" if miss else ""

import time
runs, good = [], 0

def save():
    json.dump({"name": OUT_NAME, "commit": DATA_COMMIT, "tries": TRIES, "runs": runs},
              open("eval.json", "w", encoding = "utf-8"), ensure_ascii = False, indent = 1)
    try:
        api.upload_file(path_or_fileobj = "eval.json", path_in_repo = "eval/outputs.json", repo_id = REPO_ID, commit_message = f"eval {len(runs)}")
    except Exception as e:
        print("재 보기 결과를 못 올렸어요:", type(e).__name__, e)
    if DRIVE: shutil.copy("eval.json", f"{DRIVE}/{EVAL_FILE}")

if TRAIN and not EVAL_DONE:
    for t in tests:
        for k in range(TRIES):
            t0 = time.time()
            story = write(t["prompt"], len(t["words"]))
            why = form(t["words"], story)
            good += not why
            runs.append({"set": t["set"], "try": k + 1, "words": [w[0] for w in t["words"]], "form": why, "raw": story,
                         "ms": round((time.time() - t0) * 1000)})
            print(f"\n[묶음 {t['set']} · {k + 1}번째] {'○' if not why else '✕ ' + why} · {time.time() - t0:.0f}초\n" + story, flush = True)
        save()
    print(f"\n꼴 맞음: {good}/{len(runs)}편")
    print("\n재 보기를 올렸어요: https://huggingface.co/" + REPO_ID + "/blob/main/eval/outputs.json"
          + (f"\n(구글 드라이브 '내 드라이브'의 {EVAL_FILE}에도 있어요)" if DRIVE else "")
          + ("\n이어서 크롬용으로 바꿔요 (20~30분)" if MAKE_MLC else ""))
else:
    print("재 보기는 건너뛰어요" + (" (이미 올려 두었어요)" if EVAL_DONE else ""))

## 8. 합친 모델 저장 (16비트, 약 10GB)

In [ ]:
if MAKE_MLC:
    if MERGE_SCALE != 1.0:
        # 학습한 만큼(LoRA B)을 MERGE_SCALE배로 (올려 둔 학습 파일은 그대로예요)
        n = 0
        for name, m in model.named_modules():
            if hasattr(m, "lora_B"):
                for k in m.lora_B: m.lora_B[k].weight.data *= MERGE_SCALE; n += 1
        print(f"LoRA {n}곳을 {MERGE_SCALE}배로")
    model.save_pretrained_merged("merged", tokenizer, save_method = "merged_16bit")

## 9. 크롬용으로 바꾸기
앱이 쓰는 크롬용 모델 파일에서 글 층의 가중치만 새로 눌러 담아요 (`tools/finetune/mlc_patch.py`). 모델 라이브러리(.wasm)는 앱 것 그대로 써요.

In [ ]:
if MAKE_MLC:
    import numpy as np, os, shutil
    run("rm", "-rf", "mlc_out")
    run(sys.executable, "repo/tools/finetune/mlc_patch.py", "--hf", "merged", "--mlc", "mlc_base", "--out", "mlc_out")

    # 확인: 조금만 바뀌었어야 해요 (모두 같으면 학습이 안 들어간 것, 많이 다르면 이름·꼴이 어긋난 것)
    tc = {t["name"]: (sh["dataPath"], t) for sh in json.load(open("mlc_base/tensor-cache.json"))["records"] for t in sh["records"]}
    def nibbles(folder, name):
        path, t = tc[name]
        with open(os.path.join(folder, path), "rb") as f:
            f.seek(t["byteOffset"]); a = np.frombuffer(f.read(t["nbytes"]), dtype=np.uint32)
        return np.stack([(a >> (4 * j)) & 15 for j in range(8)])
    same = [float((nibbles("mlc_base", n) == nibbles("mlc_out", n)).mean())
            for n in ["language_model.layers.0.self_attn.q_proj.q_weight", "language_model.layers.10.mlp.down_proj.q_weight", "language_model.layers.30.mlp.gate_up_proj.q_weight"]]
    print("원본과 같은 4비트 값 비율:", [f"{s:.3f}" for s in same])
    assert all(0.5 < s < 1.0 for s in same), "바뀐 정도가 이상해요 (1.0이면 학습이 안 들어감, 0.5 밑이면 이름·꼴이 어긋남). 이 줄을 Claude에게 보여 주세요"

## 10. 허깅페이스에 공개로 올리기

In [ ]:
if MAKE_MLC:
    import urllib.request
    open("mlc_out/LICENSE", "w").write(urllib.request.urlopen("https://www.apache.org/licenses/LICENSE-2.0.txt").read().decode())
    open("mlc_out/README.md", "w").write(README)
    api.upload_folder(folder_path = "mlc_out", repo_id = REPO_ID, commit_message = f"Word-story model v4, merged at scale {MERGE_SCALE} (MLC q4f16_1)")
    print("\n다 됐어요. 이 주소를 Claude에게 알려 주세요:\n  https://huggingface.co/" + REPO_ID)
else:
    print("\nMAKE_MLC = False: 크롬용 모델은 만들지 않았어요. 이 주소를 Claude에게 알려 주세요:\n  https://huggingface.co/" + REPO_ID)